# 教程: 数据预处理 - 处理缺失数据

## 目的
在实际工作中，我们得到的观测数据（如降雨、流量）几乎不可避免地会存在缺失值（Missing Values）。模型无法处理包含缺失值（在pandas中通常表示为`NaN` - Not a Number）的输入。因此，在进行模拟之前，必须对缺失数据进行预处理。

本教程将展示几种常用的处理时间序列中缺失数据的方法。

此笔记本将：
1.  加载一个包含缺失值的降雨数据CSV文件。
2.  演示并解释几种不同的填充策略：
    - **向前填充 (Forward Fill)**
    - **向后填充 (Backward Fill)**
    - **线性插值 (Linear Interpolation)**
    - **常量填充 (Filling with a Constant)**
3.  通过可视化，对比不同填充策略对原始时间序列的影响。

## 1. 环境设置和加载数据

我们加载`data/rainfall_with_missing.csv`文件。Pandas在读取时会自动将CSV中的空白字段识别为`NaN`。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df_original = pd.read_csv('../data/rainfall_with_missing.csv', index_col='date', parse_dates=True)

print("加载的原始数据 (包含NaN):")
display(df_original)

## 2. 应用不同的填充策略

现在我们创建原始数据的几个副本，并对每个副本应用一种不同的填充策略。

In [ ]:
# 策略1: 向前填充 (用前一个有效值填充)
df_ffill = df_original.fillna(method='ffill')

# 策略2: 向后填充 (用后一个有效值填充)
df_bfill = df_original.fillna(method='bfill')

# 策略3: 线性插值 (在前后两个有效值之间画一条直线)
df_interp = df_original.interpolate(method='linear')

# 策略4: 用0填充 (对于降雨数据，这通常是合理的假设)
df_zero = df_original.fillna(0)

print("--- 向前填充 (ffill) ---")
display(df_ffill)
print("\n--- 线性插值 (interp) ---")
display(df_interp)

## 3. 可视化对比

我们将原始数据（带缺口）和四种不同填充策略的结果绘制在一张图上，以便清晰地看到它们之间的差异。

In [ ]:
plt.figure(figsize=(15, 8))

# 绘制原始数据，NaN值会造成线条断裂
plt.plot(df_original['rainfall_mm'], 'ko', markersize=10, label='Original Data')

# 绘制不同策略的结果
plt.plot(df_ffill['rainfall_mm'], 'o--', label='Forward Fill')
plt.plot(df_bfill['rainfall_mm'], 's:', label='Backward Fill')
plt.plot(df_interp['rainfall_mm'], '^-', label='Linear Interpolation')
plt.plot(df_zero['rainfall_mm'], '*-.', label='Fill with Zero')

plt.title('Comparison of Missing Data Handling Strategies')
plt.xlabel('Date')
plt.ylabel('Rainfall (mm)')
plt.legend()
plt.grid(True, linestyle='--')
plt.show()

### 4. 讨论

从上图中，我们可以看到不同策略的特点：
- **向前/向后填充**: 产生了“阶梯状”的填充效果。这种方法适用于那些我们假设其值在短时间内保持不变的变量。
- **线性插值**: 产生了平滑的过渡。这种方法适用于那些我们假设其变化是渐进的变量（如水位、温度）。对于降雨这种通常是脉冲式的变量，线性插值可能会产生不符合物理实际的“小雨”。
- **用0填充**: 对于降雨数据，假设缺失的时刻没有下雨，通常是一种安全且常见的做法。

**结论**: 不存在一种“最好”的缺失数据处理方法。方法的选择取决于变量的物理特性和我们对数据的先验知识。对于不同的水文变量，应采用不同的、最合理的填充策略。